<div style="background-color: #f8f9fa; padding: 20px; border-radius: 10px; box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);">
  <div style="display: flex; justify-content: space-between; align-items: center;">
    <img src="https://sigmoidal.ai/wp-content/uploads/2024/09/Academia-Sigmoidal-Light.png" alt="Academia Sigmoidal Logo" width="250" height="auto">
    <div style="text-align: right;">
<h1 style="color: #007bff; margin: 0; font-size: 24px;">Pos-Graduacao em Visao Computacional</h1>
    </div>
</div>
<hr style="border: none; height: 1px; background-color: #007bff; margin: 20px 0;">
<h3 style="color: #343a40; margin: 0; font-size: 20px;"><strong>VIS101: Fundamentos da Visao Computacional</strong></h3>
<p style="color: #6c757d; margin: 5px 0 0; font-size: 14px;"><strong>Instrutor:</strong> Carlos Melo, MSc.</p>
</div>

# Segmentação por cor

A segmentação por cor consiste em isolar, em uma imagem, apenas os pixels que pertencem a uma faixa de cor de interesse. É uma das operações mais diretas da visão computacional e serve de base para tarefas como rastreamento de objetos e contagem.

Neste notebook o material de trabalho é um vídeo em que várias bolas de cores diferentes caem ao redor de uma pessoa. O objetivo é segmentar **apenas algumas cores** no meio das demais, e o caminho para isso passa pelo espaço de cor **HSV**.

## Preparando o ambiente

A célula abaixo baixa o vídeo e uma imagem de apoio diretamente do repositório da disciplina. Não é necessário fazer upload de nada.

In [ ]:
!mkdir -p data
!wget -q https://raw.githubusercontent.com/carlosfab/visao-computacional/main/vis101/segmentacao-por-cor/data/chuva-bolas-3cores.mp4 -O data/chuva-bolas-3cores.mp4
!wget -q https://raw.githubusercontent.com/carlosfab/visao-computacional/main/vis101/segmentacao-por-cor/data/apple.jpg -O data/apple.jpg

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 12})
print('Setup pronto.')

## Um quadro do vídeo

O OpenCV lê imagens e quadros de vídeo no formato **BGR**, não RGB. Por isso, sempre que um quadro for exibido com o matplotlib, ele precisa ser convertido para RGB, ou as cores aparecem trocadas.

In [ ]:
cap = cv2.VideoCapture('data/chuva-bolas-3cores.mp4')
cap.set(cv2.CAP_PROP_POS_FRAMES, 60)
ok, frame = cap.read()
cap.release()
frame.shape

In [ ]:
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

## Por que HSV

No espaço RGB, a cor de um objeto fica espalhada pelos três canais, o que torna difícil descrever "vermelho" ou "azul" por uma regra simples. O espaço **HSV** separa a informação em três eixos independentes:

- **H (matiz):** a cor em si.
- **S (saturação):** o quanto a cor é viva.
- **V (valor):** o quanto a cor é clara.

Como o matiz isola a cor do brilho, a seleção por cor fica muito mais estável. No OpenCV, os intervalos são **H de 0 a 179** e **S, V de 0 a 255**.

In [ ]:
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
hsv.shape

## Segmentando uma cor

A função `cv2.inRange` recebe um limite inferior e um superior em HSV e devolve uma **máscara**: pixels dentro da faixa ficam brancos (255) e os demais, pretos (0).

Para a bola azul, o matiz fica em torno de 95 a 130. Os limites mínimos de saturação e valor descartam o fundo claro, que tem saturação baixa.

In [ ]:
lower_azul = np.array([95, 80, 40])
upper_azul = np.array([130, 255, 255])
mask_azul = cv2.inRange(hsv, lower_azul, upper_azul)

In [ ]:
plt.imshow(mask_azul, cmap='gray')
plt.axis('off')
plt.show()

Aplicando a máscara sobre o quadro original com `cv2.bitwise_and`, restam apenas os pixels selecionados.

In [ ]:
res_azul = cv2.bitwise_and(frame, frame, mask=mask_azul)
plt.imshow(cv2.cvtColor(res_azul, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

## Três cores ao mesmo tempo

Para segmentar mais de uma cor, define-se uma faixa por cor e combinam-se as máscaras com `cv2.bitwise_or`. Aqui o alvo são as bolas **amarela, verde e azul**. As bolas vermelha, laranja e roxa, assim como a pessoa, devem ficar de fora.

In [ ]:
faixas = {
    'amarelo': ((20, 80, 80), (35, 255, 255)),
    'verde':   ((40, 60, 40), (85, 255, 255)),
    'azul':    ((95, 80, 40), (130, 255, 255)),
}

In [ ]:
mask = np.zeros(hsv.shape[:2], np.uint8)
for lo, hi in faixas.values():
    mask = cv2.bitwise_or(mask, cv2.inRange(hsv, np.array(lo), np.array(hi)))

Uma abertura morfológica remove respingos isolados antes de aplicar a máscara.

In [ ]:
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
seg = cv2.bitwise_and(frame, frame, mask=mask)
plt.imshow(cv2.cvtColor(seg, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

Permanecem apenas as bolas das três cores escolhidas. As demais cores e a pessoa foram descartadas, o que mostra o caráter seletivo da segmentação por faixa de cor.

## Ajustando a faixa com controles

Encontrar os limites certos na mão é trabalhoso. Com `ipywidgets`, os limites de H, S e V viram controles deslizantes e a máscara é recalculada a cada ajuste. É a forma mais rápida de calibrar uma faixa nova.

In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

from ipywidgets import interact

In [ ]:
def testar_faixa(h_min, h_max, s_min, v_min):
    m = cv2.inRange(hsv, (h_min, s_min, v_min), (h_max, 255, 255))
    plt.imshow(m, cmap='gray')
    plt.axis('off')
    plt.show()

interact(testar_faixa, h_min=(0, 179, 1), h_max=(0, 179, 1),
         s_min=(0, 255, 1), v_min=(0, 255, 1));

## Do quadro ao vídeo

O mesmo procedimento se aplica a cada quadro do vídeo. A função abaixo recebe um quadro e devolve a segmentação das três cores.

In [ ]:
def segmentar(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    m = np.zeros(hsv.shape[:2], np.uint8)
    for lo, hi in faixas.values():
        m = cv2.bitwise_or(m, cv2.inRange(hsv, np.array(lo), np.array(hi)))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    return cv2.bitwise_and(frame, frame, mask=m)

Aplicando a três quadros ao longo do vídeo, é possível conferir a estabilidade da faixa em momentos diferentes.

In [ ]:
cap = cv2.VideoCapture('data/chuva-bolas-3cores.mp4')
quadros = []
for i in [30, 60, 90]:
    cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ok, f = cap.read()
    quadros.append(f)
cap.release()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, f in zip(axes, quadros):
    ax.imshow(cv2.cvtColor(segmentar(f), cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.show()